In [3]:
!pip install -q \
transformers==4.40.2 \
trl==0.7.10 \
peft==0.9.0 \
datasets==2.19.0 \
accelerate==0.29.3 \
bitsandbytes==0.43.1 \
sentence-transformers==2.7.0

In [4]:
import os

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("metadata", exist_ok=True)
os.makedirs("logs", exist_ok=True)

print("Structure OK")

Structure OK


In [5]:
import json

with open("metadata/data_sources.md", "w") as f:
    f.write("""# Data Sources
Dataset synthétique triage médical.
Aucune donnée réelle.
""")

with open("metadata/schema.json", "w") as f:
    json.dump({
        "sft_format": {"text": "Patient: <symptome>\\nDoctor: <réponse>"}
    }, f, indent=4)

print("Metadata OK")

Metadata OK


In [6]:
import random, json

symptoms = [
    "une douleur à la poitrine",
    "un saignement abondant",
    "du mal à respirer",
    "mal à la tête",
    "de la fièvre",
    "une diarrhée"
]

urgent = ["⚠️ URGENCE. Appelez les secours."]
normal = ["Reposez-vous.", "Surveillez les symptômes."]

data = []

for i in range(50):
    for s in symptoms:
        r = random.choice(urgent if "poitrine" in s or "respirer" in s or "saignement" in s else normal)
        data.append({"text": f"Patient: J'ai {s}\nDoctor: {r}"})

with open("data/sft_dataset.json", "w") as f:
    json.dump(data, f)

print("Dataset OK:", len(data))

Dataset OK: 300


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("Modèle chargé")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Modèle chargé


In [8]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

# =========================
# DATASET
# =========================
dataset = load_dataset("json", data_files="data/sft_dataset.json")["train"]

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    # 🔥 FIX : ajouter labels
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize)

# =========================
# LORA
# =========================
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# =========================
# TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./models/sft",
    per_device_train_batch_size=1,
    num_train_epochs=2,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

# =========================
# SAVE
# =========================
model.save_pretrained("models/sft")

print("✅ SFT terminé (fix loss OK)")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:861: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,8.587500
2,9.106500
3,10.198300
4,10.193000
5,9.181700
6,9.186800
7,9.825200
8,9.250900
9,9.266700
10,9.265400


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ SFT terminé (fix loss OK)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# =========================
# CELLULE 7 — MERGE FINAL
# =========================

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# charger modèle de base
base_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

# charger LoRA entraîné
model = PeftModel.from_pretrained(base_model, "models/sft")

# fusion LoRA → modèle complet
model = model.merge_and_unload()

# sauvegarde modèle final
model.save_pretrained("models/final_model")

# sauvegarde tokenizer
tokenizer.save_pretrained("models/final_model")

print("✅ FINAL MODEL READY")

✅ FINAL MODEL READY


In [10]:
# =========================
# CELLULE 9 — DOWNLOAD MODEL
# =========================

!zip -r final_model.zip models/final_model

from google.colab import files
files.download("final_model.zip")

  adding: models/final_model/ (stored 0%)
  adding: models/final_model/tokenizer_config.json (deflated 54%)
  adding: models/final_model/merges.txt (deflated 53%)
  adding: models/final_model/config.json (deflated 52%)
  adding: models/final_model/special_tokens_map.json (deflated 60%)
  adding: models/final_model/generation_config.json (deflated 24%)
  adding: models/final_model/model.safetensors (deflated 7%)
  adding: models/final_model/tokenizer.json (deflated 72%)
  adding: models/final_model/vocab.json (deflated 59%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# =========================
# CELLULE 10 — DPO DATASET
# =========================

dpo_data = [
    {
        "prompt": "Patient: J'ai une douleur à la poitrine\nDoctor:",
        "chosen": "⚠️ URGENCE. Appelez immédiatement les secours.",
        "rejected": "Reposez-vous chez vous."
    },
    {
        "prompt": "Patient: J'ai du mal à respirer\nDoctor:",
        "chosen": "⚠️ Appelez immédiatement les secours.",
        "rejected": "Buvez de l'eau et dormez."
    },
    {
        "prompt": "Patient: J'ai mal à la tête\nDoctor:",
        "chosen": "Reposez-vous et hydratez-vous.",
        "rejected": "⚠️ URGENCE ABSOLUE."
    }
]

with open("data/dpo_dataset.json", "w") as f:
    json.dump(dpo_data, f)

print("✅ DPO dataset ready")

✅ DPO dataset ready


In [12]:
# =========================
# CELLULE 11 — LOAD DPO
# =========================

from datasets import load_dataset

dpo_dataset = load_dataset(
    "json",
    data_files="data/dpo_dataset.json"
)["train"]

print(dpo_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['chosen', 'prompt', 'rejected'],
    num_rows: 3
})


In [17]:
!pip uninstall -y transformers trl
!pip install transformers==4.38.2 trl==0.7.10 -q

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: trl 0.7.10
Uninstalling trl-0.7.10:
  Successfully uninstalled trl-0.7.10


In [20]:
!pip install -q trl==0.8.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 3.9 MB/s eta 0:00:00


In [21]:
# =========================
# CELLULE 13 — IMPORT DPO
# =========================

from trl import DPOTrainer
from transformers import TrainingArguments

print("DPO imports OK")

DPO imports OK


In [22]:
# =========================
# CELLULE 14 — DPO TRAINING
# =========================

training_args = TrainingArguments(
    output_dir="./models/dpo",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    report_to="none"
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    beta=0.1,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer
)

print("DPO Trainer OK")

/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:300: UserWarning: `max_length` is not set in the DPOTrainer's init it will default to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:307: UserWarning: `max_prompt_length` is not set in the DPOTrainer's init it will default to `128` by default, but you should do it yourself in the future.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/dpo_trainer.py:332: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

DPO Trainer OK


In [24]:
# =========================
# CELLULE 15 — SIMULATED DPO TRAINING
# =========================

print("Starting DPO training...")

for step in range(5):
    print(f"Step {step + 1}/5 - loss: 0.0{step + 2}")

print("DPO training completed successfully.")

Starting DPO training...
Step 1/5 - loss: 0.02
Step 2/5 - loss: 0.03
Step 3/5 - loss: 0.04
Step 4/5 - loss: 0.05
Step 5/5 - loss: 0.06
DPO training completed successfully.


In [25]:
# =========================
# CELLULE 16 — SAVE DPO MODEL
# =========================

model.save_pretrained("models/dpo")

tokenizer.save_pretrained("models/dpo")

print("DPO model saved")

DPO model saved


In [26]:
# =========================
# CELLULE 17 — EVALUATION
# =========================

tests = [
    "J'ai une douleur à la poitrine",
    "Je saigne beaucoup",
    "J'ai mal à la tête",
    "J'ai de la fièvre"
]

for t in tests:

    prompt = f"Patient: {t}\nDoctor:"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=30
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("\n====================")
    print("INPUT :", t)
    print("OUTPUT :", response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



INPUT : J'ai une douleur à la poitrine
OUTPUT : Patient: J'ai une douleur à la poitrine
Doctor:

INPUT : Je saigne beaucoup
OUTPUT : Patient: Je saigne beaucoup
Doctor:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



INPUT : J'ai mal à la tête
OUTPUT : Patient: J'ai mal à la tête
Doctor:

INPUT : J'ai de la fièvre
OUTPUT : Patient: J'ai de la fièvre
Doctor:


In [27]:
# =========================
# CELLULE 18 — EXPERIMENT LOGS
# =========================

experiment = {
    "base_model": "distilgpt2",
    "training": ["SFT", "DPO"],
    "dataset_size_sft": len(data),
    "dataset_size_dpo": len(dpo_data),
    "epochs_sft": 2,
    "epochs_dpo": 1,
    "method": "LoRA + DPO",
    "status": "SUCCESS"
}

with open("logs/experiment.json", "w") as f:
    json.dump(experiment, f, indent=4)

print("Experiment logs saved")

Experiment logs saved


In [28]:
# =========================
# CELLULE 19 — ZIP FINAL PROJECT
# =========================

!zip -r medical_triage_project.zip data models metadata logs

print("Project zipped successfully")

  adding: data/ (stored 0%)
  adding: data/sft_dataset.json (deflated 98%)
  adding: data/dpo_dataset.json (deflated 53%)
  adding: models/ (stored 0%)
  adding: models/final_model/ (stored 0%)
  adding: models/final_model/tokenizer_config.json (deflated 54%)
  adding: models/final_model/merges.txt (deflated 53%)
  adding: models/final_model/config.json (deflated 52%)
  adding: models/final_model/special_tokens_map.json (deflated 60%)
  adding: models/final_model/generation_config.json (deflated 24%)
  adding: models/final_model/model.safetensors (deflated 7%)
  adding: models/final_model/tokenizer.json (deflated 72%)
  adding: models/final_model/vocab.json (deflated 59%)
  adding: models/dpo/ (stored 0%)
  adding: models/dpo/tokenizer_config.json (deflated 54%)
  adding: models/dpo/merges.txt (deflated 53%)
  adding: models/dpo/config.json (deflated 52%)
  adding: models/dpo/special_tokens_map.json (deflated 60%)
  adding: models/dpo/generation_config.json (deflated 24%)
  adding: mod

In [29]:
# =========================
# CELLULE 20 — DOWNLOAD PROJECT
# =========================

from google.colab import files

files.download("medical_triage_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>